# Word Lookups: language model → word boundaries

Unsegmented characters → scratch BPE causal LM trained in both directions → surprisal at character offsets + PMI → train-only threshold → DP with punctuation, length prior and accessor-variety spans → BMES.

**Điểm đã đo ở các version Kaggle trước:** R08 Boundary F1 public.9150/private.9171, từR07 .9097/.9113. R09 BPE-dropout NLL giảm nhưng F1.9145/.9168 thấp hơn.

**Bài học:** NLL không phải segmentation F1. Một cụm xuất hiện trong nhiều ngữ cảnh trái/phải là tín hiệu lexical bổ sung. RUN_ID7/8 cô lập span bonus; RUN_ID8/9 kiểm tra dropout. Không dùng test gold.

Notebook này chứa source đầy đủ nhưng chưa execute tại bản export này; execution_count=null và outputs rỗng là có chủ ý. Không suy diễn metric mới từ notebook chưa chạy.

Bật GPU T4 trên Kaggle, attach competition tương ứng trước Run All. Loader tự tìm cả `/kaggle/input/competitions/<slug>` và `/kaggle/input/<slug>`. Code assert T4, không tự chuyển CPU. Thực hành sau contest, không dùng trong thời gian cuộc thi cấm trợ giúp.


# Word Lookups: từ biên BPE đến từ ngôn ngữ

Train không có nhãn segmentation. R07 kết hợp hai chiều, PMI, normalization, DP và punctuation đạt Boundary F1 .9097/.9113. Không có gold để phân loại lỗi test từng câu; chỉ có thể đặt giả thuyết từ cơ chế: biên BPE là đơn vị nén, không bắt buộc là biên từ; một bigram PMI không đủ nhận biết một cụm ba/bốn ký tự dùng độc lập. Không gọi những câu đoán sai chưa biết là error analysis có nhãn.

| Hạng | Paper | Điểm cần học | Quyết định / điều kiện |
|---|---|---|---|
| 1 | [Accessor Variety Criteria for Chinese Word Extraction, Feng et al., CL 2004](https://aclanthology.org/J04-1004.pdf) §4 | Cụm có nhiều predecessor/successor khác nhau có dấu hiệu dùng độc lập; AV=min(left,right) | R08 dùng AV làm bonus cho span2–4. Khai thác train corpus, không dùng từ điển ngoài |
| 2 | [Unsupervised Neural Word Segmentation for Chinese via Segmental Language Modeling, Sun & Deng, EMNLP 2018](https://aclanthology.org/D18-1531.pdf) §2.4 | Training cộng xác suất mọi segmentation bằng DP, decoding tìm segmentation tốt nhất | R08 vẫn heuristic span score; không gọi DP decoder của nó là SLM đã marginalize latent words |
| 3 | [Learning to Discover, Ground and Use Words with Segmental Neural Language Models, Kawakami et al., ACL 2019](https://aclanthology.org/P19-1645.pdf) §3–4 | Lexical memory + character decoder; regularizer ngăn lexicon học nguyên câu | Chưa cài memory generative. Giới hạn span và tần suất trong R08 là hypothesis riêng, không thay thế objective bài gốc |
| 4 | [Contextual Dependencies in Unsupervised Word Segmentation, Goldwater et al., COLING-ACL 2006](https://aclanthology.org/P06-1085.pdf) §3–4 | Word-to-word context và inference procedure ảnh hưởng undersegmentation; unigram objective có thể thích ghép collocation | Không dùng Gibbs/HDP trong lượt này; dùng để kiểm tra nguy cơ AV làm ghép quá tay |
| 5 | [BPE-Dropout, Provilkov et al., ACL 2020](https://aclanthology.org/2020.acl-main.170.pdf) §3 | Bỏ ngẫu nhiên một số merge khi train giúp model không lệ thuộc một cách chia subword | R09 dropout .1 khi train, standard BPE khi val/infer. BLEU của NMT không thể đem làm dự báo F1 segmentation |

**R08:** giữ R07; đếm substring2–4 trên 90% training fold, frequency≥5, AV≥2; sentence boundary được tính theo số lần xuất hiện đầu/cuối câu. Bonus span=min(.5,.2 log(1+AV)) cộng vào DP. Ngưỡng cố định trước scoring, không scan leaderboard. Nhãn BMES vẫn offset-safe, punctuation giữ biên bắt buộc.

**R09:** giữ R08, BPE-dropout .1 và rematerialize tokenization mỗi epoch; tắt dropout khi validation/inference. Tokenizers có RNG merge riêng nên seed42 không cam kết tái lập bitwise qua phiên bản thư viện. Ghi rõ điều này. Không nhìn nhãn test, không chọn threshold từ leaderboard.

Mỗi config tự train LM từ đầu, tối đa8 epoch/240 giây rồi decode9272 câu. Dự kiến khoảng4–5 phút/config theo R07, thêm startup mỗi batch. Chấm chính bằng Kaggle Boundary F1; NLL không phải segmentation F1.


In [ ]:
SOURCE_COMMIT = '84e9a9efaa6346578162fa12915a8368f18e6485'
RUN_ID = 8


## Shared utilities

Seed, BPE, masking, data paths, schema checks và report writer.


In [ ]:
"""Utilities for controlled, idea-driven AICC NLP experiments."""
import ast, csv, hashlib, importlib.metadata, json, math, os, random, time
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda:0")


def start_run(task, run_id, config):
    assert torch.cuda.is_available(), "GPU T4 is required; do not silently fall back to CPU"
    name = torch.cuda.get_device_name(0)
    assert "T4" in name, "Expected T4, received " + name
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    info = dict(task=task, run_id=f"R{run_id:02d}", config=config, seed=SEED,
                device=name, gpu_count=torch.cuda.device_count(), started=time.time(),
                source_commit=globals().get("SOURCE_COMMIT"),
                versions={p: importlib.metadata.version(p) for p in
                          ["torch", "numpy", "pandas", "scikit-learn", "tokenizers"]})
    print("RUN_START " + json.dumps(info), flush=True)
    return info


def find_data(slug):
    for root in (Path("/kaggle/input/competitions")/slug, Path("/kaggle/input")/slug):
        if root.is_dir():
            return root
    raise FileNotFoundError("Attach competition data: " + slug)


def make_bpe(texts, size=2048):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    tokenizer.train_from_iterator(texts, trainers.BpeTrainer(
        vocab_size=size, min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return tokenizer


def encode_texts(tokenizer, texts, max_length):
    values = np.zeros((len(texts), max_length), dtype=np.int64)
    for i, encoded in enumerate(tokenizer.encode_batch(texts)):
        ids = encoded.ids[:max_length] or [1]
        values[i, :len(ids)] = ids
    return values


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finish_run(info, validation, submission, checks, history, artifacts=None, limitations=None):
    assert checks.get("ids_unique") is True and checks.get("schema_valid") is True
    assert checks.get("row_count", 0) > 0
    submission.to_csv(OUT/"submission.csv", index=False)
    info.update(status="completed", elapsed_seconds=time.time()-info.pop("started"),
                validation=validation, submission_checks=checks, history=history,
                peak_gpu_memory_mb=torch.cuda.max_memory_allocated()/1024**2,
                artifacts=artifacts or {}, limitations=limitations or [],
                leaderboard_public=None, leaderboard_private=None)
    (OUT/"run_report.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("NLP_RUN_RESULT " + json.dumps(info), flush=True)
    print("NLP_RUN_PASSED", flush=True)
    return info


class PositionEncoder(nn.Module):
    def __init__(self, vocab, dim=96, layers=2, max_length=384, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, dim*3, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids):
        mask = ids.eq(0)
        x = self.embedding(ids) + self.position(torch.arange(ids.shape[1], device=ids.device))
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.norm(x), mask


def pools(hidden, mask):
    valid = (~mask).unsqueeze(-1)
    mean = (hidden*valid).sum(1)/valid.sum(1).clamp(min=1)
    maximum = hidden.masked_fill(~valid, -1e4).amax(1)
    return torch.cat([mean, maximum], -1)



## Model và experiments

Các config Rxx nằm đầu source. Đổi RUN_ID trước khi Run All; mỗi lượt phải bắt đầu model/optimizer mới.


In [ ]:
"""Word Lookups: scratch BPE language models and offset-safe BMES decoding."""
from collections import Counter
import unicodedata

WORD_CONFIGS={
 1:dict(idea="BPE causal Transformer surprisal mapped back to character offsets",reverse=False,normalize=False,cohesion=False,dp=False,punct=False),
 2:dict(idea="Bidirectional surprisal sees boundaries from both contexts",reverse=True,normalize=False,cohesion=False,dp=False,punct=False),
 3:dict(idea="Sentence-level normalization tests surprisal scale shift",reverse=False,normalize=True,cohesion=False,dp=False,punct=False),
 4:dict(idea="Train-only character cohesion supplements BPE-internal boundaries",reverse=False,normalize=False,cohesion=True,dp=False,punct=False),
 5:dict(idea="Dynamic programming adds a word-length prior",reverse=False,normalize=False,cohesion=False,dp=True,punct=False),
 6:dict(idea="Explicit punctuation boundaries enforce consistent BMES",reverse=False,normalize=False,cohesion=False,dp=False,punct=True),
 7:dict(idea="Combine bidirectional surprise, cohesion, DP and punctuation",reverse=True,normalize=True,cohesion=True,dp=True,punct=True)
}
WORD_CONFIGS[8]={**WORD_CONFIGS[7],"idea":"Accessor-variety lexical spans supplement boundary surprise","accessor":True}
WORD_CONFIGS[9]={**WORD_CONFIGS[8],"idea":"Accessor spans plus BPE-dropout regularized scratch LM","bpe_dropout":.1}


class BoundaryLanguageModel(nn.Module):
    def __init__(self,vocab,dim=96):
        super().__init__()
        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)
        self.position=nn.Embedding(128,dim)
        block=nn.TransformerEncoderLayer(dim,4,dim*3,dropout=0.1,batch_first=True,
                                         activation="gelu",norm_first=True)
        self.encoder=nn.TransformerEncoder(block,2,enable_nested_tensor=False)
        self.head=nn.Linear(dim,vocab)
    def forward(self,ids):
        x=self.embedding(ids)+self.position(torch.arange(ids.shape[1],device=ids.device))
        causal=torch.ones(ids.shape[1],ids.shape[1],device=ids.device,dtype=torch.bool).triu(1)
        x=self.encoder(x,mask=causal,src_key_padding_mask=ids.eq(0))
        return self.head(x)


def as_chars(value):
    if isinstance(value,str) and value.lstrip().startswith("["):
        value=ast.literal_eval(value)
    return list(value) if not isinstance(value,str) else list(value)


def tags_from_boundaries(n,boundaries):
    tags=[];start=0
    for end in sorted(set(boundaries)|{n}):
        size=end-start
        if size<=0:continue
        tags.extend(["S"] if size==1 else ["B"]+["M"]*(size-2)+["E"])
        start=end
    assert len(tags)==n
    return tags


def run_word(run_id):
    cfg={**WORD_CONFIGS[run_id],"bpe_vocab":8192,"epochs":8,"budget_seconds":240,"max_length":96}
    info=start_run("word-lookups",run_id,cfg)
    root=find_data("aicc-round-9-word-lookups")
    train=pd.read_csv(root/"train.csv");test=pd.read_csv(root/"test.csv")
    train_text=["".join(as_chars(v)) for v in train.chars]
    test_chars=[as_chars(v) for v in test.chars];test_text=["".join(c) for c in test_chars]
    indices=np.random.default_rng(SEED).permutation(len(train_text))
    cut=max(1,int(len(indices)*0.9));tr_idx,va_idx=indices[:cut],indices[cut:]
    corpus=[train_text[i] for i in tr_idx]
    fit_text=corpus+[s[::-1] for s in corpus] if cfg["reverse"] else corpus
    tokenizer=make_bpe(fit_text,cfg["bpe_vocab"])
    tokenizer.add_special_tokens(["[FWD]","[BWD]"])
    fwd_id,bwd_id=tokenizer.token_to_id("[FWD]"),tokenizer.token_to_id("[BWD]")
    tokenizer.save(str(OUT/"tokenizer.json"))
    def lm_ids(texts,reverse=False):
        x=np.zeros((len(texts),cfg["max_length"]),dtype=np.int64)
        for i,text in enumerate(texts):
            ids=[bwd_id if reverse else fwd_id]+tokenizer.encode(text[::-1] if reverse else text).ids+[3]
            ids=ids[:cfg["max_length"]]
            x[i,:len(ids)]=ids
        return x
    tr=lm_ids(corpus)
    if cfg["reverse"]:tr=np.concatenate([tr,lm_ids(corpus,True)],0)
    va=lm_ids([train_text[i] for i in va_idx])
    loader=DataLoader(TensorDataset(torch.from_numpy(tr)),batch_size=96,shuffle=True)
    model=BoundaryLanguageModel(tokenizer.get_vocab_size()).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=0.01)
    scaler=torch.amp.GradScaler("cuda")
    history=[];best=float("inf");best_state=None;tick=time.time()
    def lm_validation():
        model.eval();total=0.;count=0
        with torch.inference_mode():
            for (x,) in DataLoader(TensorDataset(torch.from_numpy(va)),batch_size=128):
                x=x.to(DEVICE)
                with torch.autocast("cuda",dtype=torch.float16):
                    logits=model(x[:,:-1])
                    loss=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),x[:,1:].reshape(-1),ignore_index=0,reduction="sum")
                total+=float(loss);count+=int(x[:,1:].ne(0).sum())
        return total/max(count,1)
    for epoch in range(cfg["epochs"]):
        if cfg.get('bpe_dropout'):
            tokenizer.model.dropout=cfg['bpe_dropout']
            try:
                tr=lm_ids(corpus)
                if cfg['reverse']:tr=np.concatenate([tr,lm_ids(corpus,True)],0)
            finally:tokenizer.model.dropout=None
            loader=DataLoader(TensorDataset(torch.from_numpy(tr)),batch_size=96,shuffle=True)
        model.train();losses=[]
        for (x,) in loader:
            x=x.to(DEVICE);optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=torch.float16):
                logits=model(x[:,:-1])
                loss=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),x[:,1:].reshape(-1),ignore_index=0)
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),1);scaler.step(optimizer);scaler.update()
            losses.append(float(loss.detach()))
            if time.time()-tick>cfg["budget_seconds"]:break
        nll=lm_validation();record=dict(epoch=epoch+1,train_nll=float(np.mean(losses)),validation_nll=nll)
        history.append(record);print("EPOCH "+json.dumps(record),flush=True)
        if nll<best:
            best=nll;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if time.time()-tick>cfg["budget_seconds"]:break
    model.load_state_dict(best_state);model.eval()
    unigram=Counter(c for s in corpus for c in s)
    bigram=Counter(pair for s in corpus for pair in zip(s,s[1:]))
    corpus_count=sum(unigram.values())
    bpe_lengths=[end-start for s in corpus[:4000] for start,end in tokenizer.encode(s).offsets if end>start]
    mean_length=float(np.clip(np.mean(bpe_lengths),1.4,3.0))
    lexical_bonus={}
    if cfg.get('accessor'):
        frequency=Counter(s[i:i+k] for s in corpus for k in [2,3,4] for i in range(max(0,len(s)-k+1)))
        retained={s for s,c in frequency.items() if c>=5}
        left={s:set() for s in retained};right={s:set() for s in retained}
        starts=Counter();ends=Counter()
        for s in corpus:
            for k in [2,3,4]:
                for i in range(max(0,len(s)-k+1)):
                    word=s[i:i+k]
                    if word not in retained:continue
                    if i:left[word].add(s[i-1])
                    else:starts[word]+=1
                    if i+k<len(s):right[word].add(s[i+k])
                    else:ends[word]+=1
        for word in retained:
            av=min(len(left[word])+starts[word],len(right[word])+ends[word])
            if av>=2:lexical_bonus[word]=min(.5,.2*math.log1p(av))
        del frequency,left,right,retained
        print('ACCESSOR_LEXICON '+json.dumps(dict(entries=len(lexical_bonus),min_frequency=5,max_span=4,weight=.2,cap=.5)),flush=True)
    @torch.inference_mode()
    def token_surprise(text,reverse=False):
        raw=text[::-1] if reverse else text
        encoded=tokenizer.encode(raw)
        values=np.zeros(max(0,len(text)-1),dtype=np.float32)
        seen=np.zeros(len(values),dtype=bool)
        # Sliding chunks avoid dropping the tail of long documents.
        tokens=encoded.ids;offsets=encoded.offsets
        stride=cfg["max_length"]-2
        for begin in range(0,len(tokens),stride):
            end=min(len(tokens),begin+stride)
            sequence=[bwd_id if reverse else fwd_id]+tokens[begin:end]
            x=torch.tensor(sequence,device=DEVICE)[None]
            with torch.autocast("cuda",dtype=torch.float16):logits=model(x).float()
            for j in range(begin+1,end):
                pos=offsets[j][0]
                if reverse:pos=len(text)-pos
                if 1<=pos<len(text):
                    local=j-begin
                    score=-float(logits[0,local].log_softmax(-1)[tokens[j]])
                    values[pos-1]=score;seen[pos-1]=True
        if seen.any():
            fallback=float(values[seen].quantile(0.25)) if hasattr(values[seen],"quantile") else float(np.quantile(values[seen],0.25))
            values[~seen]=fallback-1
        return values
    def boundary_score(text):
        score=token_surprise(text)
        if cfg["reverse"]:score=(score+token_surprise(text,True))/2
        if len(score):
            if cfg["normalize"]:score=(score-score.mean())/max(score.std(),0.1)
            if cfg["cohesion"]:
                pmi=np.array([math.log((bigram[a,b]+0.5)*corpus_count/((unigram[a]+1)*(unigram[b]+1))) for a,b in zip(text,text[1:])])
                pmi=(pmi-pmi.mean())/max(pmi.std(),0.1)
                score=(score-score.mean())/max(score.std(),0.1)-0.6*pmi
        return score
    sampled=np.concatenate([boundary_score(s) for s in corpus[:256] if len(s)>1])
    threshold=float(np.quantile(sampled,1-1/mean_length))
    def decode(text):
        n=len(text)
        if n<2:return ["S"]*n
        score=boundary_score(text)
        forced={i for i in range(1,n) if unicodedata.category(text[i-1]).startswith("P") or unicodedata.category(text[i]).startswith("P")} if cfg["punct"] else set()
        if not cfg["dp"]:boundaries=set(np.flatnonzero(score>=threshold)+1)
        else:
            probability=1/(1+np.exp(-np.clip((score-threshold)/max(sampled.std(),0.1),-8,8)))
            dp=np.full(n+1,-np.inf);prev=np.zeros(n+1,dtype=int);dp[0]=0
            for end in range(1,n+1):
                for start in range(max(0,end-6),end):
                    if any(start<b<end for b in forced):continue
                    length=end-start
                    interior=float(np.log1p(-probability[start:end-1]).sum())
                    boundary=float(math.log(probability[end-1])) if end<n else 0.
                    value=dp[start]+interior+boundary-0.08*(length-mean_length)**2+lexical_bonus.get(text[start:end],0.)
                    if value>dp[end]:dp[end]=value;prev[end]=start
            boundaries=set();end=n
            while end>0:
                if end<n:boundaries.add(end)
                end=int(prev[end])
        boundaries|=forced
        return tags_from_boundaries(n,boundaries)
    predictions=[]
    for i,text in enumerate(test_text):
        tags=decode(text);assert len(tags)==len(test_chars[i])
        predictions.append(str(tags))
        if i%1000==0:print("DECODE "+str(i),flush=True)
    sub=pd.DataFrame(dict(id=test.id,bio_tags=predictions))
    checks=dict(ids_unique=sub.id.is_unique,schema_valid=list(sub.columns)==["id","bio_tags"],
                row_count=len(sub),lengths_match=True,valid_bmes=True)
    torch.save(best_state,OUT/"model.pt")
    return finish_run(info,dict(metric="BPE_LM_NLL_proxy",value=best,direction="minimize",
          samples=len(va_idx),split="train_only_10pct",boundary_f1=None),sub,checks,history,
          artifacts=dict(tokenizer="tokenizer.json",pretrained_weights=False,train_only_statistics=True,
                         threshold=threshold,mean_bpe_length=mean_length,accessor_lexicon_entries=len(lexical_bonus)),
          limitations=["LM NLL is a proxy, not segmentation F1; no segmentation gold labels were used.",
                       "BPE boundaries and linguistic words differ; decoding is a testable hypothesis.",
                       "Only training text is used for vocabulary, LM fitting and threshold calibration.",
                       "Accessor bonus is a transfer hypothesis, not a full segmental neural LM or Bayesian word model.",
                       "BPE-dropout uses tokenizers stochastic merges; seed42 alone does not guarantee identical merge samples across library versions."])



In [ ]:
OUT = Path('/kaggle/working') / 'word-lookups' / f'R{RUN_ID:02d}'
OUT.mkdir(parents=True, exist_ok=True)
report = run_word(RUN_ID)
print('Submission:', OUT / 'submission.csv')
